In [ ]:
!pip install -q groq gradio

import os, json
from getpass import getpass
from groq import Groq
import gradio as gr

# ---------- API key ----------
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("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")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Paste your Groq API key: ")

client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"  # alt: "llama-3.1-8b-instant" (faster/cheaper)

SYSTEM_PROMPT = """You are an expert nutritionist agent.
Given a food (plus optional serving size, health conditions, allergies), reply with ONLY a JSON object:
{
  "food": "clean name of the food",
  "edible": true or false,
  "verdict": "SAFE TO EAT" | "EAT IN MODERATION" | "AVOID",
  "reason": "1-2 sentence explanation of the verdict",
  "serving": "serving size used",
  "calories_kcal": number,
  "protein_g": number,
  "carbs_g": number,
  "fat_g": number,
  "fiber_g": number,
  "sugar_g": number,
  "sodium_mg": number,
  "vitamins_minerals": ["top 3-5 nutrients"],
  "benefits": ["1-3 short health benefits"],
  "warnings": ["allergy / condition / safety warnings, empty list if none"],
  "tip": "one practical tip"
}
Rules:
- If the input is not food, or is toxic/non-edible (soap, plastic, poisonous mushroom...), set edible=false,
  verdict="AVOID", all numbers 0, and explain why.
- If no serving is given, assume a typical single serving (or 100 g) and state it in "serving".
- Take the user's health conditions and allergies into account for the verdict and warnings.
- Values are approximate estimates."""


def ask_agent(food, serving, conditions, allergies):
    user_msg = (
        f"Food: {food}\nServing: {serving or 'not specified'}\n"
        f"Health conditions: {conditions or 'none'}\nAllergies: {allergies or 'none'}"
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_msg}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


ICONS = {"SAFE TO EAT": "✅", "EAT IN MODERATION": "⚠️", "AVOID": "❌"}


def analyze(food, serving, conditions, allergies):
    if not food or not food.strip():
        return "### Please enter a food name.", "", ""
    try:
        d = ask_agent(food.strip(), serving, conditions, allergies)
    except Exception as e:
        return f"### Error\n`{e}`", "", ""

    verdict = d.get("verdict", "EAT IN MODERATION")
    icon = ICONS.get(verdict, "ℹ️")
    head = (f"## {icon} {verdict}\n"
            f"*{d.get('food', food)}* — {d.get('serving', '')}\n\n"
            f"{d.get('reason', '')}")

    table = (
        "| Nutrient | Amount |\n|---|---|\n"
        f"| 🔥 Calories | *{d.get('calories_kcal', 0)} kcal* |\n"
        f"| 💪 Protein | {d.get('protein_g', 0)} g |\n"
        f"| 🍞 Carbs | {d.get('carbs_g', 0)} g |\n"
        f"| 🥑 Fat | {d.get('fat_g', 0)} g |\n"
        f"| 🌾 Fiber | {d.get('fiber_g', 0)} g |\n"
        f"| 🍬 Sugar | {d.get('sugar_g', 0)} g |\n"
        f"| 🧂 Sodium | {d.get('sodium_mg', 0)} mg |"
    )

    def bullets(items):
        return "\n".join(f"- {i}" for i in items) if items else "- None"

    extra = (
        f"### 🧬 Key nutrients\n{bullets(d.get('vitamins_minerals'))}\n\n"
        f"### 🌿 Benefits\n{bullets(d.get('benefits'))}\n\n"
        f"### 🚨 Warnings\n{bullets(d.get('warnings'))}\n\n"
        f"### 💡 Tip\n{d.get('tip', '')}\n\n"
        "<sub>Estimates only — not medical advice. Consult a doctor or dietitian for health conditions.</sub>"
    )
    return head, table, extra


with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Nutrition AI Agent") as demo:
    gr.Markdown("# 🥗 Nutrition AI Agent\nEnter a food to get calories, nutrition facts, and whether it's safe to eat.")
    with gr.Row():
        with gr.Column(scale=1):
            food = gr.Textbox(label="Food", placeholder="e.g. banana, biryani, 2 idlis, raw egg")
            serving = gr.Textbox(label="Serving size (optional)", placeholder="e.g. 1 cup, 200 g, 2 pieces")
            conditions = gr.Textbox(label="Health conditions (optional)", placeholder="e.g. diabetes, high BP")
            allergies = gr.Textbox(label="Allergies (optional)", placeholder="e.g. peanuts, lactose")
            btn = gr.Button("Analyze 🔍", variant="primary")
            gr.Examples(
                examples=[["Banana", "1 medium", "", ""],
                          ["Chicken biryani", "1 plate", "diabetes", ""],
                          ["Peanut butter toast", "2 slices", "", "peanuts"],
                          ["Soap", "", "", ""]],
                inputs=[food, serving, conditions, allergies],
            )
        with gr.Column(scale=1):
            verdict_out = gr.Markdown()
            table_out = gr.Markdown()
            extra_out = gr.Markdown()

    btn.click(analyze, [food, serving, conditions, allergies], [verdict_out, table_out, extra_out])
    food.submit(analyze, [food, serving, conditions, allergies], [verdict_out, table_out, extra_out])

demo.launch(share=True, debug=True)

Paste your Groq API key: ··········


/tmp/ipykernel_879/3182519272.py:102: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Nutrition AI Agent") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://245ec293b7e6ac6aa4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
